# scikit-learn

A refresher on **scikit-learn (`sklearn`)** — the workhorse library for *classical* machine learning in Python: regression, classification, clustering, dimensionality reduction, and the glue (preprocessing, pipelines, cross-validation, model selection) that holds a modeling workflow together.

**Domain:** AI/ML Tooling  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

**What it is.** scikit-learn is a single, consistent library covering the *classical* ML toolkit on top of NumPy/SciPy: linear and tree-based models, SVMs, k-NN, naive Bayes, ensembles (random forests, gradient boosting), clustering (k-means, DBSCAN), PCA/manifold learning, plus the surrounding machinery — feature scaling, encoding, imputation, train/test splitting, cross-validation, hyperparameter search, and metrics.

**The problem it solves.** Before sklearn, every model came from a different package with a different API, and stitching preprocessing + model + evaluation together was bespoke glue code that leaked test data into training. scikit-learn standardized everything behind one tiny interface (`fit` / `predict` / `transform`) and gave you `Pipeline` + `cross_val_score` so the *entire* workflow — not just the model — is fit only on training data. That makes experiments reproducible and leakage-resistant by construction.

**When to reach for it.**
- Tabular data (rows × columns), small-to-medium scale (fits in RAM, up to ~millions of rows).
- You want a strong baseline fast, or the problem genuinely is classical ML (fraud scoring, churn, demand forecasting, tabular classification).
- You need preprocessing, model selection, and honest evaluation in one coherent API.

**When *not* to.**
- Deep learning on images/text/audio → PyTorch/TensorFlow/JAX (sklearn has no GPU, no autograd, no neural net training beyond a basic MLP).
- Massive out-of-core / distributed data → Spark MLlib, or sklearn's partial-fit / Dask-ML.
- State-of-the-art *tabular* accuracy → gradient-boosting specialists (XGBoost, LightGBM, CatBoost) usually edge out sklearn's own boosting, though they copy its API.

## 2. Mental Model

**Everything is an estimator with the same three verbs.**

```
            ┌─────────────────────────────────────────────┐
   X, y ──▶ │  estimator.fit(X, y)   ← learns from data    │
            └─────────────────────────────────────────────┘
                         │ stored as  attribute_  (trailing underscore)
                         ▼
   Predictors:   estimator.predict(X_new)      → labels / values
   Transformers: estimator.transform(X_new)    → new feature matrix
   Scorers:      estimator.score(X, y)         → a default metric
```

- A **predictor** (model) implements `fit` + `predict` (e.g. `LogisticRegression`).
- A **transformer** (preprocessing) implements `fit` + `transform` (e.g. `StandardScaler`, `OneHotEncoder`).
- A **`Pipeline`** chains transformers then a final predictor and *is itself an estimator* — so it too has `fit`/`predict`. Calling `pipe.fit(X_train)` fits every step on training data only; `pipe.predict(X_test)` applies the *same* learned transforms before predicting. This is the single most important idea: it makes the pipeline the unit you cross-validate and tune, so preprocessing can never peek at the test fold.

Anything learned from data is stored as an attribute ending in `_` (e.g. `scaler.mean_`, `model.coef_`). If it has a trailing underscore, it came from `fit`.

## 3. Key Concepts

- **Estimator** — any object with `fit`. The base abstraction.
- **`fit` / `predict` / `transform` / `fit_transform`** — the universal API. `fit_transform` = fit then transform in one call (fit on train; use plain `transform` on test).
- **Hyperparameters vs. learned parameters** — hyperparameters are constructor args you set (`C`, `max_depth`, `n_estimators`); learned parameters are the `*_` attributes set by `fit`. `get_params()` / `set_params()` expose hyperparameters.
- **`Pipeline`** — chains steps; named steps let you address nested hyperparameters as `stepname__param` (double underscore).
- **`ColumnTransformer`** — applies different transformers to different columns (e.g. scale numerics, one-hot encode categoricals) and concatenates the result. The standard entry point for mixed tabular data.
- **Cross-validation** — `cross_val_score` / `cross_validate` / `KFold` / `StratifiedKFold` estimate generalization by rotating which fold is held out. The honest alternative to a single train/test split.
- **Model selection** — `GridSearchCV` / `RandomizedSearchCV` wrap an estimator, search hyperparameters by cross-validation, and refit the best config on all the data. The result is itself an estimator.
- **Metrics & scoring** — `sklearn.metrics` (accuracy, precision/recall/F1, ROC-AUC, RMSE, R²). Pass a metric by string name (`scoring="f1_macro"`) or build one with `make_scorer`.
- **`random_state`** — seeds the RNG for splits and stochastic models; set it for reproducible results.

## 4. Setup

scikit-learn is pure-Python + NumPy/SciPy wheels — no GPU, no special toolchain. One install covers everything.

```bash
pip install scikit-learn      # pulls in numpy, scipy, joblib, threadpoolctl
```

It ships with small toy datasets (`load_iris`, `load_wine`, `load_diabetes`) and synthetic data generators (`make_classification`, `make_blobs`) so every example below runs offline with no downloads.

In [1]:
# Setup: confirm the install. (Uncomment the pip line on a fresh environment.)
# %pip install -q scikit-learn

import sklearn
import numpy as np

print("scikit-learn", sklearn.__version__)
print("numpy", np.__version__)

scikit-learn 1.9.0
numpy 2.5.0


## 5. Worked Examples

### Example 1 — A leakage-free classification Pipeline

The canonical workflow: load data, split, build a `Pipeline` (scale → model), fit, and evaluate on the held-out test set. Scaling lives *inside* the pipeline so its statistics are learned from the training fold only — never the test set.

In [2]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0
)

# Scaler + model as one estimator: fit() learns scaling on TRAIN only.
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)

pred = pipe.predict(X_test)
print(f"test accuracy: {accuracy_score(y_test, pred):.3f}\n")
print(classification_report(y_test, pred, target_names=load_wine().target_names))

test accuracy: 1.000

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        15
     class_1       1.00      1.00      1.00        18
     class_2       1.00      1.00      1.00        12

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



### Example 2 — Cross-validation + hyperparameter search

A single split is a noisy estimate. `cross_val_score` rotates the held-out fold for an honest mean. Then `GridSearchCV` searches hyperparameters *by cross-validation* and refits the winner on all the training data — note the `clf__C` syntax addressing the pipeline step's parameter.

In [3]:
from sklearn.model_selection import cross_val_score, GridSearchCV

# 5-fold stratified CV (the default for classifiers) on the whole pipeline.
cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
print(f"CV accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

# Tune the regularization strength C of the LogisticRegression step.
grid = GridSearchCV(
    pipe,
    param_grid={"clf__C": [0.01, 0.1, 1.0, 10.0]},
    cv=5,
    scoring="accuracy",
)
grid.fit(X_train, y_train)

print(f"best C: {grid.best_params_['clf__C']}")
print(f"best CV score: {grid.best_score_:.3f}")
print(f"held-out test score (refit best): {grid.score(X_test, y_test):.3f}")

CV accuracy: 0.970 +/- 0.015
best C: 0.1
best CV score: 0.970
held-out test score (refit best): 1.000


### Example 3 — Mixed columns with `ColumnTransformer`

Real tabular data mixes numeric and categorical features. `ColumnTransformer` routes each column group to the right transformer and concatenates the output — all still inside one pipeline you can fit, cross-validate, and tune as a unit.

In [4]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Tiny synthetic table: 2 numeric cols + 1 categorical col.
rng = np.random.RandomState(0)
n = 200
num = rng.normal(size=(n, 2))
cat = rng.choice(["red", "green", "blue"], size=n).reshape(-1, 1)
Xmix = np.hstack([num, cat]).astype(object)
ymix = ((num[:, 0] + (cat[:, 0] == "red")) > 0).astype(int)

pre = ColumnTransformer([
    ("num", StandardScaler(), [0, 1]),                       # scale numeric cols
    ("cat", OneHotEncoder(handle_unknown="ignore"), [2]),    # encode the category
])
model = Pipeline([("pre", pre), ("rf", RandomForestClassifier(n_estimators=100, random_state=0))])

scores = cross_val_score(model, Xmix, ymix, cv=5)
print(f"mixed-feature pipeline CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

mixed-feature pipeline CV accuracy: 0.995 +/- 0.010


## 6. Gotchas & Pitfalls

- **Data leakage from fitting on everything.** Calling `scaler.fit_transform(X)` *before* the split (or any `fit` outside the pipeline) lets test-set statistics bleed into training, inflating scores. Fix: put every transformer in a `Pipeline` and fit the pipeline inside cross-validation.
- **`fit_transform` on test data.** On the test set use `transform`, never `fit_transform` — the latter re-learns parameters from test data. Pipelines handle this for you; the trap is manual preprocessing.
- **Unscaled features for distance/gradient models.** SVMs, k-NN, k-means, and regularized linear models assume comparable feature scales. Forgetting `StandardScaler` quietly wrecks them. (Tree-based models are scale-invariant, so don't bother there.)
- **Imbalanced classes + accuracy.** 95% "accuracy" is meaningless when 95% of rows are one class. Use `stratify=` in the split, `StratifiedKFold`, and metrics like F1 / ROC-AUC / `balanced_accuracy`; consider `class_weight="balanced"`.
- **`random_state` left unset.** Splits and stochastic models vary run to run; set `random_state` everywhere you want reproducibility.
- **`LogisticRegression` convergence warnings.** "max_iter reached" means it didn't converge — scale your features and/or raise `max_iter`.
- **Sparse vs. dense surprises.** `OneHotEncoder` returns a sparse matrix by default; some downstream steps want dense. Pass `sparse_output=False` if needed.
- **Reaching for deep learning too early.** For tabular problems a tuned `RandomForest` or gradient-boosting model usually beats a neural net with a fraction of the effort. Exhaust classical ML first.

## 7. When to Use vs Alternatives

| Need | Reach for | Why |
| --- | --- | --- |
| Tabular ML, in-RAM, strong baseline fast | **scikit-learn** | One consistent API, batteries included, leakage-safe pipelines |
| Best-in-class tabular accuracy (boosting) | **XGBoost / LightGBM / CatBoost** | Faster, more accurate gradient boosting; native categorical & missing handling — and they mimic the sklearn API |
| Images / text / audio, GPU, custom nets | **PyTorch / TensorFlow / JAX** | Autograd + GPU + deep architectures; sklearn has no GPU and only a toy MLP |
| Data too big for RAM / distributed | **Spark MLlib, Dask-ML, cuML** | Out-of-core or cluster/GPU execution; sklearn is single-machine (some `partial_fit` streaming) |
| Classical stats with inference (p-values, CIs) | **statsmodels** | sklearn optimizes prediction, not statistical inference |

**Rule of thumb:** start in scikit-learn for any tabular problem — it gives you a clean baseline and honest evaluation in an hour. Graduate to gradient-boosting libraries for the last few points of tabular accuracy, and to deep-learning frameworks only when the data is unstructured (pixels, tokens, waveforms). The sklearn API has become the de-facto standard, so even when you move on, `fit`/`predict`/`Pipeline` come with you.

## 8. Resources

- **Official docs & user guide** — https://scikit-learn.org/stable/user_guide.html (the user guide is genuinely excellent; read the relevant chapter before tuning blindly).
- **API reference** — https://scikit-learn.org/stable/api/index.html
- **Examples gallery** — https://scikit-learn.org/stable/auto_examples/index.html (runnable, copy-pasteable recipes for nearly every estimator).
- **"Choosing the right estimator" cheat-sheet** — https://scikit-learn.org/stable/machine_learning_map.html
- **Common pitfalls & recommended practices** — https://scikit-learn.org/stable/common_pitfalls.html (the canonical write-up on data leakage and how pipelines prevent it).